In [ ]:
# group 6, topic 2

In [6]:
#Part A
import numpy as np
import pandas as pd

#basic parameter
total_time = 120          # the total time: 2h = 120 min
arrival_rate = 2          # arrival rate
sojourn_mean = 8          # mean food selection time in minutes
sojourn_std = 2           # standard deviation of food selection time
service_mean = 1          # mean cashier service time in minutes
num_simulations = 1000    #number of simulation replications for part A(b)

#convert lognormal mean and standard deviation
variance = sojourn_std ** 2
mu_log = np.log(sojourn_mean**2 / np.sqrt(sojourn_mean**2 + variance))
sigma_log = np.sqrt(np.log(1 + variance / sojourn_mean**2))

def run_simulation():
    #generate customer arrival times
    arrival_times = []
    current_time = 0
    while current_time < total_time:
        # arrival time follow exponential distribution
        inter_arrival = np.random.exponential(1 / arrival_rate)
        current_time = current_time + inter_arrival

        if current_time <= total_time:
            arrival_times.append(current_time)

    num_customers = len(arrival_times)

    # generate meal selection time and queue entry time
    sojourn_durations = np.random.lognormal(mean=mu_log, sigma=sigma_log, size=num_customers)
    # generate customers enter the cashier queue times after finishing food selection
    queue_entry_times = np.array(arrival_times) + sojourn_durations
    # generate cashier service times
    service_times = np.random.exponential(scale=service_mean, size=num_customers)
    
    #initialiaze event time array and state variables
    waiting_times = np.zeros(num_customers)
    service_start_times = np.zeros(num_customers)
    departure_times = np.zeros(num_customers)
    
    # time when cashier becomes free
    cashier_free_at = [0.0, 0.0]

    # customers enter the cashier queue according to queue entry time
    queue_entry_order = np.argsort(queue_entry_times) #argost() returns the index order from smallest to largest

    #process customers one by one
    for customer_index in queue_entry_order:
        queue_entry_time = queue_entry_times[customer_index]

        # choose earlier free cashier, if both free, first use cashier[0]
        if cashier_free_at[0] <= cashier_free_at[1]:
            chosen_cashier = 0
        else:
            chosen_cashier = 1

        # service can only start after both conditions are met:
        #1. the customer has entered the queue
        #2. the chosen cashier is avalible
        service_start_time = max(queue_entry_time, cashier_free_at[chosen_cashier])
        
        #record event times for this customer
        service_start_times[customer_index] = service_start_time
        waiting_times[customer_index] = service_start_time - queue_entry_time
        departure_times[customer_index] = service_start_time + service_times[customer_index]

        # update the chosen cashier's next free time
        cashier_free_at[chosen_cashier] = departure_times[customer_index]

  #store results
    results = pd.DataFrame({
        "Customer_ID": np.arange(1, num_customers + 1),
        "Arrival_Time": arrival_times,
        "Sojourn_Time": sojourn_durations,
        "Waiting_Time": waiting_times,
        "Departure_Time": departure_times,
        "Queue_Entry_Time": queue_entry_times,
        "Service_Start_Time": service_start_times,
        "Service_Time": service_times
    })
    return results


In [ ]:
#Part A (a): one simulation run
df_results = run_simulation()
print(df_results.to_string(index=False))
print("\ntotal customer:")
print(len(df_results))

In [22]:
#part A (b): Monte Carlo estimation
mean_waiting_times = []

for i in range(num_simulations):
    simulation_result = run_simulation()
    mean_waiting_times.append(simulation_result["Waiting_Time"].mean())

mean_waiting_times = np.array(mean_waiting_times)

#Monte Carlo estimator
mc_estimator = np.mean(mean_waiting_times)

#standard deviation of simulation estimates
mc_std = np.std(mean_waiting_times, ddof=1)

# 95% confidence interval
ci_lower = mc_estimator - 1.96 * mc_std / np.sqrt(num_simulations)
ci_upper = mc_estimator + 1.96 * mc_std / np.sqrt(num_simulations)

print(f"Monte Carlo expected time: {mc_estimator:.4f} minutes")
print(f"Standard Deviation: {mc_std:.4f} minutes")
print(f"95% Confidence Interval: ({ci_lower:.4f}, {ci_upper:.4f}) minutes")

Monte Carlo expected time: 5.1570 minutes
Standard Deviation: 3.4958 minutes
95% Confidence Interval: (4.9403, 5.3737) minutes


In [ ]:
#Part B simulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def run_part_b_simulation():
    #parameters
    total_time = 60           # 12:00-13:00
    sojourn_mean = 2.0        # average meal selection time
    sojourn_std = 1.0         # meal selection std
    cashier_factors = [0.8, 1.0, 1.3] # cashiers' different efficiency
    num_cashiers = 3

    # lognormal parameters transformer
    v_raw = sojourn_std ** 2
    mu_log = np.log(sojourn_mean**2 / np.sqrt(sojourn_mean**2 + v_raw))
    sigma_log = np.sqrt(np.log(1 + v_raw / sojourn_mean**2))

    # NHPP arrival time generation
    arrival_times = []
    
    # period1:12:00-12:40 (0-40 min), so many students so choose lambda = 7
    t = 0
    while True:
        t += np.random.exponential(1 / 7)
        if t < 40: arrival_times.append(t)
        else: break         
    # period2:12:40-13:00 (40-60 min), students go to class, lambda = 1.5
    t = 40
    while True: 
        t += np.random.exponential(1 / 1.5)
        if t < 60: arrival_times.append(t)
        else: break        
    num_customers = len(arrival_times)
    # costumer property generation
    customer_types = np.random.choice(['Regular', 'Large-order'], size=num_customers, p=[0.8, 0.2])
    base_service_times = np.where(customer_types == 'Regular',
                                  np.random.exponential(scale=1.0, size=num_customers),
                                  np.random.exponential(scale=3.0, size=num_customers))
    
    # sojourn means meal selection
    sojourn_durations = np.random.lognormal(mean=mu_log, sigma=sigma_log, size=num_customers)
    queue_entry_times = np.array(arrival_times) + sojourn_durations

    # core loop
    queue_entry_order = np.argsort(queue_entry_times) # np.argsort is used to find the index, so queue_entry_order's elements are index which can help to find student
    cashier_free_at = [0.0, 0.0, 0.0]# cashiers' initial status
    active_departures = []
    results_list = []

    for idx in queue_entry_order:
        entry_time = queue_entry_times[idx]
        c_type = customer_types[idx]
        base_svc = base_service_times[idx]
        # for each idx, update the length of current queue for the students come to see
        active_departures = [d for d in active_departures if d > entry_time]
        current_queue_length = len(active_departures)
        # balking with probability in different stages
        balking_prob = 0.0
        if current_queue_length >= 20: balking_prob = 1.0
        elif current_queue_length >= 15: balking_prob = 0.6
        elif current_queue_length >= 10: balking_prob = 0.2
        # random.rand() is used to get a number in [0,1) and if it jumps into [0,balking_prob), means this student won't buy meal
        if np.random.rand() < balking_prob:
            results_list.append({
                "Customer_ID": idx + 1,
                "Type": c_type,
                "Arrival_Time": arrival_times[idx],
                "Queue_Entry": entry_time,
                "Service_Start": entry_time, 
                "Waiting_Time": 0.0,
                "Chosen_Cashier": "None",
                "Status": "Lost"
            })
            continue

        # cashier status update
        idle_cashiers = [i for i in range(num_cashiers) if cashier_free_at[i] <= entry_time]#find all the free cashier
        chosen_cashier = min(idle_cashiers) if idle_cashiers else np.argmin(cashier_free_at)#choose which cashier to serve, argmin find index of the min value
        # if some cashiers free find the one whose index smaller, if all busy, find the cloest, earliest 
        service_start = max(entry_time, cashier_free_at[chosen_cashier])
        #cashiers have different efficiency which has set before
        actual_service_time = base_svc * cashier_factors[chosen_cashier]
        waiting_time = service_start - entry_time
        departure_time = service_start + actual_service_time
        cashier_free_at[chosen_cashier] = departure_time
        active_departures.append(departure_time)
        #append the student who buy meal
        results_list.append({
            "Customer_ID": idx + 1,
            "Type": c_type,
            "Arrival_Time": arrival_times[idx],
            "Queue_Entry": entry_time,
            "Service_Start": service_start,
            "Waiting_Time": waiting_time,
            "Chosen_Cashier": f"Cashier {chosen_cashier + 1}",
            "Status": "Served"
        })

    return pd.DataFrame(results_list)
df_part_B = run_part_b_simulation()
served_df = df_part_B[df_part_B["Status"] == "Served"]

# data visualization
total_arrivals = len(df_part_B)
served_customers = len(served_df)
lost_customers = total_arrivals - served_customers
lost_rate = (lost_customers / total_arrivals) * 100 if total_arrivals > 0 else 0

# for the most time the lost rate is high, but i think this result is similar to bnbu's happy time food choosen cafeteria
print("---- part B ----")
print(f"total arrival: {total_arrivals} people")
print(f"real customer: {served_customers} people")
print(f"customer lost): {lost_customers} people (lost rate: {lost_rate:.2f}%)")
print("\n--- waiting time for real customers ---")
print(f"average waiting time: {served_df['Waiting_Time'].mean():.2f} min")
print(f"longest waiting time: {served_df['Waiting_Time'].max():.2f} min")
print("\n--- cashier data ---")
cashier_counts = served_df['Chosen_Cashier'].value_counts().sort_index()
for cashier, count in cashier_counts.items():
    print(f"{cashier} dealing customers number: {count} people")


df_results = run_part_b_simulation()
print(df_results.to_string(index=False))
  




In [9]:
#part B Monte Carlo estimation
mean_waiting_times = []
balking_rates = []
num_simulations=1000
for i in range(num_simulations):
    simulation_result = run_part_b_simulation()
    served_customers = simulation_result[simulation_result["Status"] == "Served"]
    total_arrivals = len(simulation_result)
    served_count = len(served_customers)
    lost_count = total_arrivals - served_count
        
    if served_count > 0:
        mean_waiting_times.append(served_customers['Waiting_Time'].mean())
    if total_arrivals > 0:
        balking_rates.append((lost_count / total_arrivals) * 100)

mean_waiting_times = np.array(mean_waiting_times)

#Monte Carlo estimator
mc_estimator = np.mean(mean_waiting_times)
mc_est_balk = np.mean(balking_rates)
#standard deviation of simulation estimates
mc_std = np.std(mean_waiting_times, ddof=1)

std_balk = np.std(balking_rates, ddof=1)
se_balk = std_balk / np.sqrt(num_simulations)
# 95% confidence interval
ci_lower = mc_estimator - 1.96 * mc_std / np.sqrt(num_simulations)
ci_upper = mc_estimator + 1.96 * mc_std / np.sqrt(num_simulations)
ci_balk = (mc_est_balk - 1.96 * se_balk, mc_est_balk + 1.96 * se_balk)
# waitting time
print(f"Monte Carlo expected time: {mc_estimator:.4f} minutes")
print(f"Standard Deviation: {mc_std:.4f} minutes")
print(f"95% Confidence Interval: ({ci_lower:.4f}, {ci_upper:.4f}) minutes")
# costumers lost rate
print(f"Monte Carlo Point Estimate (Mean): {mc_est_balk:.2f}%")
print(f"Sample Standard Deviation (Std):   {std_balk:.2f}%")
print(f"Standard Error of Estimator (SE):  {se_balk:.2f}")
print(f"95% Confidence Interval (95% CI):  [{ci_balk[0]:.2f}%, {ci_balk[1]:.2f}%]")

Monte Carlo expected time: 5.3805 minutes
Standard Deviation: 0.8929 minutes
95% Confidence Interval: (5.3252, 5.4359) minutes
Monte Carlo Point Estimate (Mean): 58.16%
Sample Standard Deviation (Std):   4.62%
Standard Error of Estimator (SE):  0.15
95% Confidence Interval (95% CI):  [57.87%, 58.45%]
